# Linear Probing 평가 — STL10 / CIFAR10

**순서**
1. Cell 1: Drive 마운트
2. Cell 2: 코드 복사 + 패키지 설치
3. Cell 3: Feature 추출 (backbone → .npy)
4. Cell 4: evaluate.py 실행 (linear probing)

MoCo v2 / MoCo v3 각각 Cell 3~4 실행하면 됨.

In [ ]:
# Cell 1 — Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — 코드 복사 + 패키지 설치
import os, subprocess, shutil

DRIVE_DIR = '/content/drive/MyDrive/ssl_project'
WORK_DIR  = '/content/ssl_project'

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)

for item in ['ssl_lib', 'scripts', 'configs', 'requirements.txt', 'setup.py']:
    src, dst = f'{DRIVE_DIR}/{item}', f'{WORK_DIR}/{item}'
    shutil.copytree(src, dst) if os.path.isdir(src) else shutil.copy2(src, dst)

# evaluate.py 도 복사
shutil.copy2(f'{DRIVE_DIR}/evaluate.py', f'{WORK_DIR}/evaluate.py')

os.chdir(WORK_DIR)

subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)

# data / outputs / logs → Drive 심링크
for d in ['data', 'outputs', 'logs']:
    remote = f'{DRIVE_DIR}/{d}'
    local  = f'{WORK_DIR}/{d}'
    os.makedirs(remote, exist_ok=True)
    if os.path.exists(local) or os.path.islink(local):
        os.remove(local) if os.path.islink(local) else shutil.rmtree(local)
    os.symlink(remote, local)

print('Setup 완료!')
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Cell 3 — Feature 추출
# METHOD 변경: 'mocov2' 또는 'mocov3'
import glob, os, subprocess

METHOD = 'mocov2'   # ← 여기만 바꾸면 됨

OUTPUT_DIRS = {
    'mocov2': 'outputs/mocov2_r50_seed42',
    'mocov3': 'outputs/mocov3_vits_seed42',
}
CONFIG_FILES = {
    'mocov2': 'configs/mocov2_r50.yaml',
    'mocov3': 'configs/mocov3_vits.yaml',
}

os.chdir('/content/ssl_project')

# 마지막 backbone 체크포인트 자동 탐색
backbone_ckpts = sorted(
    glob.glob(f'{OUTPUT_DIRS[METHOD]}/backbone_ep*.pth'),
    key=lambda p: int(p.split('backbone_ep')[1].replace('.pth', ''))
)
if not backbone_ckpts:
    raise FileNotFoundError(f'{OUTPUT_DIRS[METHOD]}/ 에 backbone_ep*.pth 파일이 없습니다.')

backbone_path = backbone_ckpts[-1]
feature_dir   = f'outputs/{METHOD}_features'

print(f'Method   : {METHOD}')
print(f'Backbone : {backbone_path}')
print(f'Feature  : {feature_dir}/')
print()

proc = subprocess.Popen(
    [
        'python3', '-u', 'scripts/extract_features.py',
        '--backbone',    backbone_path,
        '--config',      CONFIG_FILES[METHOD],
        '--output-dir',  feature_dir,
        '--data-dir',    './data',
        '--batch-size',  '512',
        '--num-workers', '2',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nFeature 추출 완료 (exit={proc.returncode})')

In [ ]:
# Cell 4 — Linear Probing 평가
import os, subprocess

METHOD = 'mocov2'   # ← Cell 3 과 동일하게 맞출 것

os.chdir('/content/ssl_project')
feature_dir = f'outputs/{METHOD}_features'

proc = subprocess.Popen(
    [
        'python3', '-u', 'evaluate.py',
        '--stl10-train-features',    f'{feature_dir}/stl10_train_features.npy',
        '--stl10-train-labels',      f'{feature_dir}/stl10_train_labels.npy',
        '--stl10-test-features',     f'{feature_dir}/stl10_test_features.npy',
        '--stl10-test-labels',       f'{feature_dir}/stl10_test_labels.npy',
        '--cifar10-train-features',  f'{feature_dir}/cifar10_train_features.npy',
        '--cifar10-train-labels',    f'{feature_dir}/cifar10_train_labels.npy',
        '--cifar10-test-features',   f'{feature_dir}/cifar10_test_features.npy',
        '--cifar10-test-labels',     f'{feature_dir}/cifar10_test_labels.npy',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n평가 완료 (exit={proc.returncode})')